# compresso-recsys quickstart

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/zombak79/compresso-recsys/blob/main/examples/notebooks/quickstart.ipynb)

compresso-recsys has three parts you can use alone or together. This notebook runs all three on MovieLens 1M in about two minutes on a CPU.

1. **Data checkpoint.** One portable file with the splits, item IDs, metadata and its own provenance.
2. **Model base class.** Every model shares `fit`, `predict_on_batch`, `recommend` and `save`/`load`. A model from another library only needs a short adapter.
3. **Evaluation and statistics.** Every model is scored on the same users, followed by a paired comparison with confidence intervals and Holm-corrected p-values.

In [1]:
# Install compresso-recsys and, for the adapter example, implicit (skipped where already installed).
import importlib.util, subprocess, sys

def pip(*args):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *args])

if importlib.util.find_spec("compresso_recsys") is None:
    try:
        pip("compresso-recsys==0.4.0")
    except subprocess.CalledProcessError:  # before the PyPI release: the release branch
        pip("git+https://github.com/zombak79/compresso-recsys@release/0.4.0")
if importlib.util.find_spec("implicit") is None:
    pip("implicit")

## ① Data checkpoint

A single call downloads ML-1M and preprocesses it. It splits the users into training and held-out test users (the strong-generalisation protocol), and writes everything to one ZIP. Reading the checkpoint back gives plain scipy/numpy arrays.

In [2]:
import compresso_recsys as cr

path = cr.build_recsys_checkpoint(dataset="ml1m", split_mode="user_split", seed=42,
                                  checkpoint_path="artifacts/ml1m.zip", annotation_source="none",
                                  show_progress=False)
with cr.read_checkpoint(path) as root:
    split = cr.load_recsys_split(root)
    provenance = cr.load_provenance(root)

print("training matrix:", split["x_train"].shape, "| test users:", split["test_source_matrix"].shape[0])
print("raw sources:", [(s["name"], s["sha256"][:12]) for s in provenance["sources"]])
print("content digest:", provenance["content_sha256"][:16], "(identical across rebuilds)")

training matrix: (4533, 3260) | test users: 1000
raw sources: [('item_text_descriptions.feather', 'fffc9dae4cc7'), ('ml-1m.zip', 'a6898adb50b9')]
content digest: 2f32912b0d73deaa (identical across rebuilds)


## ② Models

Three built-in models are fitted on the training users. Passing `item_ids` makes every model speak the dataset's own movie IDs.

The fourth model is ALS from [implicit](https://github.com/benfred/implicit), another library. It joins the evaluation through a ten-line adapter that implements `predict_on_batch`.

In [3]:
x, item_ids = split["x_train"], split["train_item_ids"]
models = {
    "Pop": cr.PopularityBaseline().fit(x, item_ids=item_ids),
    "ItemKNN": cr.ItemKNNRecommender().fit(x, item_ids=item_ids),
    "EASE": cr.EASE().fit(x, item_ids=item_ids),
}

In [4]:
import numpy as np, torch
from compresso import SRPTensor
from implicit.als import AlternatingLeastSquares

class ImplicitALS:
    """Adapter: everything the evaluator needs from a model of another library."""
    def __init__(self, **params):
        self.model = AlternatingLeastSquares(**params)
    def fit(self, x):
        self.model.fit(x.tocsr().astype(np.float32), show_progress=False)
        return self
    def predict_on_batch(self, source, *, k, exclude_seen=True):
        source = source.tocsr().astype(np.float32)
        # recalculate_user=True folds in held-out users; without it, implicit would
        # silently score the training users with the same row numbers.
        ids, scores = self.model.recommend(np.arange(source.shape[0]), source, N=k,
                                           recalculate_user=True, filter_already_liked_items=exclude_seen)
        return SRPTensor(cols=torch.from_numpy(ids).long(), vals=torch.from_numpy(scores).float(),
                         shape=source.shape)

models["ALS (implicit)"] = ImplicitALS(factors=128, regularization=0.05, alpha=2.0,
                                        iterations=15, random_state=0).fit(x)

## ③ Evaluation and statistics

Every model is scored on the same held-out users, with each user's history as input. The default metrics are calibrated recall@20/50 and nDCG@100. `compare_models` then tests each model against EASE with paired bootstrap CIs and a randomization test, Holm-corrected over all comparisons.

In [5]:
results = {
    name: cr.evaluate_recommender(model, source=split["test_source_matrix"],
                                  targets=split["test_target_matrix"],
                                  sample_ids=split["test_eval_user_ids"], exclude_seen=True)
    for name, model in models.items()
}
{name: {k: round(v, 4) for k, v in r.metrics.items()} for name, r in results.items()}

{'Pop': {'calibrated_recall@20': 0.1768,
  'calibrated_recall@50': 0.248,
  'ndcg@100': 0.23},
 'ItemKNN': {'calibrated_recall@20': 0.3121,
  'calibrated_recall@50': 0.4168,
  'ndcg@100': 0.3799},
 'EASE': {'calibrated_recall@20': 0.357,
  'calibrated_recall@50': 0.475,
  'ndcg@100': 0.4303},
 'ALS (implicit)': {'calibrated_recall@20': 0.3112,
  'calibrated_recall@50': 0.4274,
  'ndcg@100': 0.383}}

In [6]:
report = cr.compare_models(results, reference="EASE", metrics=["ndcg@100", "calibrated_recall@20"])
report.summary()

,metric,baseline,candidate,baseline_mean,candidate_mean,difference,ci_low,ci_high,adjusted_p_value,significant,direction,p_floored
0,ndcg@100,EASE,Pop,0.430329,0.230034,-0.200294,-0.209586,-0.190881,0.0006,True,worse,True
1,ndcg@100,EASE,ItemKNN,0.430329,0.379856,-0.050472,-0.056407,-0.044563,0.0006,True,worse,True
2,ndcg@100,EASE,ALS (implicit),0.430329,0.383021,-0.047307,-0.052883,-0.041733,0.0006,True,worse,True
3,calibrated_recall@20,EASE,Pop,0.356974,0.176801,-0.180172,-0.192079,-0.168220,0.0006,True,worse,True
4,calibrated_recall@20,EASE,ItemKNN,0.356974,0.312067,-0.044907,-0.053869,-0.036041,0.0006,True,worse,True
5,calibrated_recall@20,EASE,ALS (implicit),0.356974,0.311159,-0.045815,-0.054210,-0.037374,0.0006,True,worse,True


In [7]:
print(report.to_latex(layout="compact"))

\begin{table}
\centering
\caption{R@$k$: calibrated recall, hits$/\min(k,|R_u|)$; 95\% paired bootstrap CIs on the per-user difference; paired randomization (sign-flip) test with $B=9{,}999$; Holm correction over 6 hypotheses; $^{\blacktriangle}$/$^{\blacktriangledown}$: significantly better/worse than EASE; $\le$: raw p at the Monte Carlo floor $1/(B{+}1)$, so the corrected value is an upper bound.}
\begin{tabular}{lrr}
\toprule
Model & nDCG@100 & R@20 \\
\midrule
EASE & \textbf{0.4303} & \textbf{0.3570} \\
\midrule
Pop & 0.2300$^{\blacktriangledown}$ & 0.1768$^{\blacktriangledown}$ \\
ItemKNN & 0.3799$^{\blacktriangledown}$ & 0.3121$^{\blacktriangledown}$ \\
ALS (implicit) & 0.3830$^{\blacktriangledown}$ & 0.3112$^{\blacktriangledown}$ \\
\bottomrule
\end{tabular}
\end{table}


## Save, reload, recommend

A fitted model is one file. Reloaded, it recommends from any list of item IDs, such as a new user's history. Seen items and a blocklist can be excluded.

In [8]:
models["EASE"].save("artifacts/ease.zip")
ease = cr.EASE.load("artifacts/ease.zip")

history = ["1", "2571", "260"]          # Toy Story, The Matrix, Star Wars IV
recs = ease.recommend([history], k=5, exclude_seen=True, blocklist=["1196"])  # not Empire

titles = split["entity_metadata"].set_index("item_id")["title"]
[(item, titles[item], round(float(score), 3)) for item, score in zip(recs.item_ids[0], recs.scores[0])]

[('3114', 'Toy Story 2 (1999)', 0.178),
 ('1210', 'Star Wars: Episode VI - Return of the Jedi (1983)', 0.143),
 ('1198', 'Raiders of the Lost Ark (1981)', 0.121),
 ('589', 'Terminator 2: Judgment Day (1991)', 0.108),
 ('2028', 'Saving Private Ryan (1998)', 0.099)]

## Next

- [Implementing a recommender](https://zombak79.github.io/compresso-recsys/implementing-a-recommender.html): a native model on the base class.
- [Comparing models statistically](https://zombak79.github.io/compresso-recsys/statistical-comparison.html): reading and reporting the comparison.
- [Reproducing the paper benchmark](https://zombak79.github.io/compresso-recsys/paper-reproduction.html): the same steps at scale.